In [ ]:
import numpy as np
from numpy import pi

from math import factorial

from scipy.special import hermite, laguerre
from scipy.interpolate import interp1d

from os import path
from PIL import Image

import os
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output


__all__ = ['SLM', 'HG', 'LG', 'AdvHG', 'CGH']


class SLM:
    device = 'HoloEye, PLUTO-2-NIR-011'

    pixel_size = 8
    resolution = (1920, 1080)
    
    x, y = np.meshgrid(
           np.arange(-resolution[0]/2, resolution[0]/2) * pixel_size,
          -np.arange(-resolution[1]/2, resolution[1]/2) * pixel_size)
    
    rho = x**2 + y**2

    norm_x = x / (resolution[0] * pixel_size)
    norm_y = y / (resolution[1] * pixel_size)



class _Mode:
    def __init__(self, n, m):
        valid = all(isinstance(x, int) and x >= 0 for x in (n, m))
        if valid:
            self.n = n
            self.m = m
        else:
            raise ValueError('orders must be positive integers')
    
    @classmethod
    def check(self, inputs):
        if not isinstance(inputs, (HG, LG, AdvHG, AdvHG_extend)):
            raise ValueError('mode must be a instance of HG, LG, AdvHG or AdvHG_extend')



class HG(_Mode):
    def complex_amplitude(self, w0):
        n, m = self.n, self.m
        norm = np.sqrt(2**(1-n-m) / (pi * factorial(m) * factorial(n))) / w0
        hx, hy= hermite(n)(2**.5 * SLM.x / w0), hermite(m)(2**.5 * SLM.y / w0)
        amplitude = norm * hx * hy * np.exp(-SLM.rho/(w0**2))

        return amplitude * np.exp(0j)
    

class AdvHG_extend:
    def __init__(self, x1, opx, x2, y1, opy, y2):
        import re

        if opx not in ('+', '-') or opy not in ('+', '-'):
            raise ValueError("Operators must be '+' or '-'")

        for val in [x1, x2, y1, y2]:
            if not (isinstance(val, int) and val >= 0):
                raise ValueError("Operands must be non-negative integers")

        self.x1 = x1
        self.x2 = x2
        self.y1 = y1
        self.y2 = y2
        self.opx = opx
        self.opy = opy

        # 运算符映射为数值
        op_map = {'+': 1, '-': -1}
        self.opx_val = op_map[opx]
        self.opy_val = op_map[opy]

        mode_x = f"{x1}{opx}{x2}"
        mode_y = f"{y1}{opy}{y2}"
        pattern = r'^\d+[\+\-]\d+$'
        if not re.match(pattern, mode_x) or not re.match(pattern, mode_y):
            raise ValueError("Invalid format: must be like '1+2', '3-1', etc.")

    def complex_amplitude(self, w0):
        hgx1y1 = HG(self.x1, self.y1).complex_amplitude(w0)
        hgx1y2 = HG(self.x1, self.y2).complex_amplitude(w0)
        hgx2y1 = HG(self.x2, self.y1).complex_amplitude(w0)
        hgx2y2 = HG(self.x2, self.y2).complex_amplitude(w0)

        return (hgx1y1 + self.opy_val * hgx1y2 + self.opx_val * hgx2y1 + self.opx_val * self.opy_val * hgx2y2) / 2      #e.g., <x|0-1>*<y|2+3> = 1/\sqrt{2}(<x|0>-<x|1>)*1/\sqrt{2}(<y|2>+<y|3>)

class LG(_Mode):
    def __init__(self):
        raise NotImplementedError('LG mode is not supported yet')



class AdvHG:
    def __init__(self, pm):
        if pm in (-1, 1):
            self.pm = pm
        else:
            raise ValueError('pm must be -1 or 1, 1 stands for + mode, -1 stands for - mode')

    def complex_amplitude(self, w0):
        hg00 = HG(0, 0).complex_amplitude(w0)
        hg01 = HG(0, 1).complex_amplitude(w0)
        return (hg00 + self.pm * hg01) / 2**.5





class CGH:
    def __init__(self, sigma, *modes, nx=500, ny=50):

        [_Mode.check(mode) for mode in modes]

        w0 = 2*sigma

        if len(modes) == 1:
            ca = modes[0].complex_amplitude(w0)*np.exp(2j*pi*SLM.norm_x*nx)

        elif len(modes) == 2:
            ca0, ca1 = [m.complex_amplitude(w0) for m in modes]
            ca = ca0*np.exp(2j*pi*SLM.norm_y*ny)*np.exp(2j*pi*SLM.norm_x*nx) + ca1*np.exp(-2j*pi*SLM.norm_y*ny)*np.exp(2j*pi*SLM.norm_x*nx)
          
        elif len(modes) > 2:
            # 手动输入参数（角度制）
            radius = float(input("请输入圆弧的半径："))
            flareangle = float(input("请输入圆弧的张角（单位：度）："))
            flareangle_rad = np.deg2rad(flareangle)  # 角度 -> 弧度
            _nx, _ny = generate_arc_coordinates(radius, flareangle_rad, len(modes))
            ca = sum(m.complex_amplitude(w0) *np.exp(2j * pi * SLM.norm_y * y) *np.exp(2j * pi * SLM.norm_x * x) for m, x, y in zip(modes, _nx, _ny))

        else:
            raise ValueError('invalid modes parameter, possibly due to ' +
                             'the number of modes being <= 0')
   

        
        fx2 = interp1d(np.linspace(0, 1, 801),np.load(path.join(os.getcwd(), 'fx2.npy')))
        a = np.abs(ca) / np.abs(ca).max()
        phi = np.angle(ca)

        _temp = fx2(a) * np.sin(phi)
        
        _temp = ((_temp - _temp.min()) / (_temp.max() - _temp.min())) * 255

        self.cgh = _temp.astype(np.uint8)
        self.img = Image.fromarray(self.cgh)

    def result(self):
        return self.cgh
    
    def show(self):
        self.img.show()
    
    def save(self, file):
        file = path.expanduser(file)
        # 始终保存（覆盖已有文件）
        self.img.save(file)



def generate_arc_coordinates(radius, flareangle, num_modes):  #生成等间距圆弧上的坐标点，圆弧位于第一二象限，关于x轴对称
    theta = np.linspace(-flareangle / 2, flareangle / 2, num_modes)
    _nx = radius * np.cos(theta)
    _ny = radius * np.sin(theta)
    return _nx, _ny




#生成一系列全息图，应用于二维多模AdvSPADE成像，例如：当num_modes=10， 则生成100张全息图，分别对应HG(0-1, 0-1), HG(0-1, 0+1), HG(0-1, 2-3), HG(0-1, 2+3)....HG(8+9, 8+9)
def generate_multimode_advhg_holograms(sigma, num_modes, file_folder='current folder'):
    # 判断num_modes是否为偶数
    if num_modes % 2 != 0:
        raise ValueError("num_modes should be even")
    
    # 目标文件夹路径处理
    if file_folder == 'current folder':
        base_path = os.path.join(os.getcwd(), 'SLM_image')
    else:
        base_path = os.path.join(os.path.expanduser(file_folder), 'SLM_image')

    os.makedirs(base_path, exist_ok=True)

    combinations = []

    # 构造组合：x=(i±i+1), y=(j±j+1)
    for i in range(0, num_modes - 1, 2):
        for sign1 in ['-', '+']:
            for j in range(0, num_modes - 1, 2):
                for sign2 in ['-', '+']:
                    combinations.append((i, sign1, i + 1, j, sign2, j + 1))

    # 按顺序生成图像并命名
    for index, (i, sign1, i2, j, sign2, j2) in enumerate(tqdm(combinations, desc='Generating holograms')):
        mode = AdvHG_extend(i, sign1, i2, j, sign2, j2)
        hologram = CGH(sigma, mode)

        # 文件名格式示例：advhg_000_(0-1,0+1).png
        filename = f"advhg_{index:03d}_({i}{sign1}{i2},{j}{sign2}{j2}).png"
        save_path = os.path.join(base_path, filename)

        hologram.save(save_path)


#生成一系列全息图，应用于二维多模StdSPADE成像，例如：当num_modes=10， 则生成100张全息图，分别对应HG(0, 1), HG(0, 2), HG(0, 3),....HG(9, 9)
def generate_multimode_stdhg_holograms(sigma, num_modes, file_folder='current folder'):
    import os
    from tqdm import tqdm

    if file_folder == 'current folder':
        base_path = os.path.join(os.getcwd(), 'SLM_image')
    else:
        base_path = os.path.join(os.path.expanduser(file_folder), 'SLM_image')

    os.makedirs(base_path, exist_ok=True)

    count = 1
    for i in tqdm(range(num_modes), desc='Generating std HG holograms (rows)'):
        for j in range(num_modes):
            mode = HG(i, j)
            hologram = CGH(sigma, mode)
            filename = f"stdhg_{count:03d}_({i},{j}).png"
            save_path = os.path.join(base_path, filename)
            hologram.img.save(save_path)
            count += 1



generate_multimode_advhg_holograms(103, 5)

ValueError: num_modes should be even

In [ ]:
w0=206
cgh = CGH(103, AdvHG_extend(3, '-', 1, 2, '+', 4))
cgh.show()



In [26]:
mode1 = HG(2, 0)
mode2 = HG(0, 2)
mode3 = HG(1, 2)
cgh = CGH(103, mode1, mode2, mode3)
cgh.show()